# Player Analytics — EDA & Data Validation
Covers every visualisation on the Player Analytics dashboard page:
- Squad composition (position, age, club, league, country)
- Club representation across all 48 squads
- Market value distributions
- Age profiles
- Historical WC goal scorers (from Fjelstul DB)
- Caps distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROCESSED = Path('../data/processed')

squads    = pd.read_csv('../data/raw/wc2026_squads.csv')
fj_goals  = pd.read_parquet(PROCESSED / 'fjelstul_goals.parquet')
fj_squads = pd.read_parquet(PROCESSED / 'fjelstul_squads.parquet')
fj_players = pd.read_parquet(PROCESSED / 'fjelstul_players.parquet')
fj_apps   = pd.read_parquet(PROCESSED / 'fjelstul_player_appearances.parquet')

from ingestion.team_name_map import WC2026_TEAMS, WC2026_TEAM_NAMES

# Filter squads to WC 2026 teams only
squads_wc = squads[squads['team_name'].isin(WC2026_TEAM_NAMES)].copy()
squads_wc['confederation'] = squads_wc['team_name'].map(WC2026_TEAMS)

print(f'2026 squads: {len(squads_wc)} players across {squads_wc["team_name"].nunique()} teams')
print(f'Fjelstul goals: {len(fj_goals)} rows across {fj_goals["tournament_id"].nunique()} tournaments')

## 1. Squad composition by position

In [ ]:
pos_counts = squads_wc['position'].value_counts().reset_index()
pos_counts.columns = ['position', 'count']

fig = px.bar(pos_counts, x='position', y='count',
             color='position',
             title='Position distribution across all WC 2026 squads',
             template='plotly_white',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

# Per team breakdown
pos_team = squads_wc.groupby(['team_name','position']).size().unstack(fill_value=0)
print('\nPosition counts per team (sample):')
display(pos_team.head(10))

## 2. Age profile — oldest / youngest squads

In [ ]:
squad_age = squads_wc.groupby('team_name')['age'].agg(['mean','min','max']).reset_index()
squad_age.columns = ['team_name','avg_age','min_age','max_age']
squad_age = squad_age.sort_values('avg_age', ascending=False)
squad_age['confederation'] = squad_age['team_name'].map(WC2026_TEAMS)

fig = px.bar(squad_age, x='team_name', y='avg_age',
             color='confederation',
             title='Average squad age — WC 2026',
             labels={'avg_age': 'Avg Age', 'team_name': 'Team'},
             template='plotly_white',
             error_y=squad_age['max_age'] - squad_age['avg_age'],
             error_y_minus=squad_age['avg_age'] - squad_age['min_age'])
fig.update_xaxes(tickangle=45)
fig.update_layout(height=500)
fig.show()

print(f'Oldest squad: {squad_age.iloc[0]["team_name"]} ({squad_age.iloc[0]["avg_age"]:.1f} avg)')
print(f'Youngest squad: {squad_age.iloc[-1]["team_name"]} ({squad_age.iloc[-1]["avg_age"]:.1f} avg)')

## 3. Age distribution (all players)

In [ ]:
fig = px.histogram(squads_wc, x='age', nbins=20,
                   color='position',
                   title='Age distribution across all WC 2026 players',
                   template='plotly_white',
                   barmode='overlay', opacity=0.7,
                   color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(legend_title='Position')
fig.show()

print(f'Overall mean age: {squads_wc["age"].mean():.1f}')
print(f'Peak age band:    {squads_wc["age"].mode()[0]}')

## 4. Club representation — which clubs have the most players?

In [ ]:
club_counts = (
    squads_wc[squads_wc['club'].notna()]
    .groupby('club').size()
    .reset_index(name='players')
    .sort_values('players', ascending=False)
    .head(25)
)

fig = px.bar(club_counts, x='players', y='club',
             orientation='h',
             title='Top 25 clubs by WC 2026 player representation',
             labels={'players': 'Players', 'club': 'Club'},
             template='plotly_white',
             color='players',
             color_continuous_scale='Blues')
fig.update_layout(height=700, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 5. Club country — where do WC 2026 players ply their trade?

In [ ]:
league_counts = (
    squads_wc[squads_wc['club_country'].notna()]
    .groupby('club_country').size()
    .reset_index(name='players')
    .sort_values('players', ascending=False)
    .head(20)
)

fig = px.bar(league_counts, x='club_country', y='players',
             title='Top 20 club leagues by WC 2026 player representation',
             labels={'players': 'Players', 'club_country': 'Club Country'},
             template='plotly_white',
             color='players',
             color_continuous_scale='Teal')
fig.update_xaxes(tickangle=45)
fig.show()

## 6. International caps — most experienced players

In [ ]:
top_caps = squads_wc.nlargest(20, 'caps')[['player_name','team_name','position','age','caps','international_goals']]

fig = px.scatter(top_caps, x='caps', y='player_name',
                 color='team_name', size='international_goals',
                 title='Top 20 most-capped players at WC 2026',
                 labels={'caps': 'International Caps', 'player_name': ''},
                 template='plotly_white')
fig.update_layout(height=600)
fig.show()

display(top_caps)

## 7. Historical WC top scorers (all tournaments, Fjelstul DB)

In [ ]:
# Non-own-goal scorers across all WC tournaments
goals_agg = (
    fj_goals[fj_goals['own_goal'] == 0]
    .groupby(['player_name', 'team_name'])['goal_id']
    .count()
    .reset_index(name='wc_goals')
    .sort_values('wc_goals', ascending=False)
    .head(20)
)

fig = px.bar(goals_agg, x='wc_goals', y='player_name',
             orientation='h', color='team_name',
             title='All-time WC top scorers (1930–2022)',
             labels={'wc_goals': 'World Cup Goals', 'player_name': ''},
             template='plotly_white')
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

display(goals_agg)

## 8. WC 2026 players with WC scoring history

In [ ]:
# Find 2026 squad players who have scored in a previous WC
prev_scorers = fj_goals[fj_goals['own_goal'] == 0].groupby('player_name')['goal_id'].count().reset_index(name='prev_wc_goals')

# Normalise names for join
squads_wc['player_name_clean'] = squads_wc['player_name'].str.strip()
merged = squads_wc.merge(prev_scorers, left_on='player_name_clean', right_on='player_name', how='inner')

print(f'{len(merged)} WC 2026 players with previous WC scoring history:')
display(merged[['team_name','player_name_x','position','age','caps','prev_wc_goals']]
        .sort_values('prev_wc_goals', ascending=False)
        .rename(columns={'player_name_x': 'player_name'}))

## 9. Squad quality heatmap — caps per position per team

In [ ]:
# Average caps by position and team
caps_heat = (
    squads_wc[squads_wc['position'].isin(['GK','DEF','MID','FWD'])]
    .groupby(['team_name','position'])['caps']
    .mean()
    .unstack(fill_value=0)
    .round(1)
)

# Sort by total caps
caps_heat['_total'] = caps_heat.sum(axis=1)
caps_heat = caps_heat.sort_values('_total', ascending=False).drop(columns=['_total'])

fig = px.imshow(
    caps_heat,
    title='Average caps by position — WC 2026 squads',
    labels={'x': 'Position', 'y': 'Team', 'color': 'Avg Caps'},
    color_continuous_scale='YlOrRd',
    aspect='auto'
)
fig.update_layout(height=1000)
fig.show()

## 10. Data quality checks

In [ ]:
print('=== SQUAD DATA QUALITY ===')
print(f'Total players:        {len(squads_wc)}')
print(f'Teams covered:        {squads_wc["team_name"].nunique()} / {len(WC2026_TEAM_NAMES)}')
print(f'Missing position:     {squads_wc["position"].isna().sum()}')
print(f'Missing age:          {squads_wc["age"].isna().sum()}')
print(f'Missing caps:         {squads_wc["caps"].isna().sum()}')
print(f'Missing club:         {squads_wc["club"].isna().sum()}')
print(f'Missing club_country: {squads_wc["club_country"].isna().sum()}')

print('\n=== SQUADS PER TEAM ===')
sizes = squads_wc.groupby('team_name').size().sort_values()
display(pd.DataFrame({'size': sizes, 'ok': sizes.between(23,26)}))

missing_teams = WC2026_TEAM_NAMES - set(squads_wc['team_name'].unique())
print(f'\nWC teams with no squad data: {sorted(missing_teams)}')